In [1]:
import os
import glob
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pull_data import (
    get_user_id_by_email,
    get_sleep,
    fetch_all_signals,
)
from process_data import (
    find_ppg_periods,
    attach_signals_per_period,
    process_block_data,
)

from process_signal import (
    predict_hr_accel,
    predict_hr_ppg
)

In [2]:
user_email = 'suhani.sharma@temple.com'
sleep_date = '2026-05-30'

user_id = get_user_id_by_email(user_email)
sleep_duration = get_sleep(user_id, sleep_date, sleep_date)
sleep_start, sleep_end = str(sleep_duration['start_time_local'][0]), str(sleep_duration['end_time_local'][0])

raw_data_path = f"sleep/raw_data/{user_email}_{sleep_start.replace(' ', 'T')}_{sleep_end.replace(' ', 'T')}"
if not os.path.exists(raw_data_path):
    raw_data = fetch_all_signals(user_id, sleep_start, sleep_end)

    # Duty-cycle densely-sampled green: keep 30 s every 5 min (30 s on, 4:30 off).
    # Green only — accel/hr untouched.
    KEEP_MS, PERIOD_MS, DENSE_GAP_MS = 30_000, 300_000, 100.0
    g_ts = raw_data.loc[raw_data['green'].notna(), 'timestamp'].to_numpy(dtype=float)
    if g_ts.size > 1 and np.median(np.diff(g_ts)) < DENSE_GAP_MS:
        t0 = g_ts[0]
        phase = (raw_data['timestamp'].to_numpy(dtype=float) - t0) % PERIOD_MS
        drop_green = raw_data['green'].notna() & (phase >= KEEP_MS)
        raw_data.loc[drop_green, 'green'] = np.nan
        # Remove rows left with no signal at all (green-only rows just nulled).
        sig_cols = ['green', 'accel_x', 'accel_y', 'accel_z', 'firmware_hr']
        sig_cols = [c for c in sig_cols if c in raw_data.columns]
        raw_data = raw_data.dropna(subset=sig_cols, how='all').reset_index(drop=True)

    raw_data.to_csv(raw_data_path, index=False)
else:
    raw_data = pd.read_csv(raw_data_path)

In [3]:
ppg_periods = find_ppg_periods(raw_data)
block_raw_data = attach_signals_per_period(ppg_periods, raw_data)

In [4]:
block_process_data = process_block_data(block_raw_data, accel_stride_s=15)

ppg_peaks_all = []
accel_peaks_all = []

for _, row in block_process_data.iterrows():
    accel_ts = np.asarray(row['accel_ts'])

    block_accel_peaks = []
    for axw, ayw in zip(row['accel_x_windows'], row['accel_y_windows']):
        ax = np.asarray(axw, dtype=float)
        ay = np.asarray(ayw, dtype=float)
        if ax.size == 0 or ay.size == 0:
            block_accel_peaks.append([])
            continue
        block_accel_peaks.append(predict_hr_accel(ax, ay))
    accel_peaks_all.append(block_accel_peaks)

    block_ppg_peaks = []
    for gw, gw_ts in zip(row['green_windows'], row['green_window_ts']):
        g = np.asarray(gw, dtype=float)
        g_ts = np.asarray(gw_ts, dtype=float)
        if g.size == 0 or g_ts.size == 0:
            block_ppg_peaks.append(np.nan)
            continue
        mask = (accel_ts >= g_ts[0]) & (accel_ts <= g_ts[-1])
        t_a = accel_ts[mask].astype(float)
        if t_a.size == 0:
            block_ppg_peaks.append(np.nan)
            continue
        block_ppg_peaks.append(predict_hr_ppg(g, g_ts, t_a))
    ppg_peaks_all.append(block_ppg_peaks)

block_process_data['ppg_peaks'] = ppg_peaks_all
block_process_data['accel_peaks'] = accel_peaks_all

In [5]:
def _trailing(s, trail):
    return s.shift(1).rolling(trail, min_periods=1).mean()

green_rows, accel_rows, hr_rows = [], [], []

for bi, (_, row) in enumerate(block_process_data.iterrows()):
    ppg_peaks = row['ppg_peaks']
    for wi in range(len(row['green_windows'])):
        green_rows.append({
            'block': bi, 'window': wi,
            'green': row['green_windows'][wi],
            'green_ts': row['green_window_ts'][wi],
            'green_mean': row['green_window_mean'][wi], 'green_std': row['green_window_std'][wi],
            'peak': ppg_peaks[wi] if wi < len(ppg_peaks) else np.nan,
        })

    for wi in range(len(row['accel_x_windows'])):
        accel_rows.append({
            'block': bi, 'window': wi,
            'accel_x': row['accel_x_windows'][wi],
            'accel_y': row['accel_y_windows'][wi],
            'accel_z': row['accel_z_windows'][wi],
            'accel_ts': row['accel_window_ts'][wi],
            'x_mean': row['accel_x_window_mean'][wi], 'x_std': row['accel_x_window_std'][wi],
            'y_mean': row['accel_y_window_mean'][wi], 'y_std': row['accel_y_window_std'][wi],
            'z_mean': row['accel_z_window_mean'][wi], 'z_std': row['accel_z_window_std'][wi],
            'peak': row['accel_peaks'][wi],
        })

    for wi in range(len(row['hr'])):
        hr_rows.append({
            'block': bi, 'window': wi,
            'hr': row['hr'][wi],
            'hr_ts': row['hr_ts'][wi],
            })

green_df = pd.DataFrame(green_rows)
g_base = _trailing(green_df['green_mean'], 100)
green_drop = (green_df['green_mean'] - g_base) > 1000 
green_df = green_df[~green_drop].reset_index(drop=True)

accel_df = pd.DataFrame(accel_rows)
ax_base, ay_base, az_base = _trailing(accel_df['x_mean'], 100), _trailing(accel_df['y_mean'], 100), _trailing(accel_df['z_mean'], 1000)
mean_jump = (((accel_df['x_mean'] - ax_base) > 500) | 
             ((accel_df['y_mean'] - ay_base) > 500) |
             ((accel_df['z_mean'] - az_base) > 500))
std_bad = (accel_df['x_std'] > 200) | (accel_df['y_std'] > 200) | (accel_df['z_std'] > 200)
accel_df = accel_df[~(mean_jump | std_bad)].reset_index(drop=True)

hr_df = pd.DataFrame(hr_rows)

In [6]:
# predict_hr_accel returns candidate peaks per window; pick the candidate
# nearest a reference HR. The reference blends the most recent reported
# firmware HR (at or before this window) with recent picks:
#   reference = 0.5 * last_reported_hr + 0.5 * mean(last 5 picked peaks)
# Recursive: each picked peak feeds the running history for the next row.
# accel_df is in block/window (chronological) order, so we iterate as-is.

hr_lookup = hr_df.dropna(subset=['hr', 'hr_ts'])
hr_ts = hr_lookup['hr_ts'].to_numpy(dtype=float)
hr_val = hr_lookup['hr'].to_numpy(dtype=float)

def _last_ts(seq):
    return seq[-1] if len(seq) else np.nan

accel_df['peak_raw'] = accel_df['peak']

picked, hist = [], []   # hist = recent finite picked peaks
for cand, ts_seq in zip(accel_df['peak_raw'], accel_df['accel_ts']):
    cand = np.asarray(cand, dtype=float)
    cand = cand[np.isfinite(cand)]
    t = _last_ts(ts_seq)

    prior = hr_ts <= t
    closest_hr = hr_val[prior][np.argmax(hr_ts[prior])] if (np.isfinite(t) and prior.any()) else np.nan
    last5 = np.mean(hist[-5:]) if hist else np.nan


    if np.isfinite(closest_hr) and np.isfinite(last5):
        ref = 0.5 * closest_hr + 0.5 * last5
    elif np.isfinite(last5):
        ref = last5
    else:
        ref = closest_hr 

    if cand.size and np.isfinite(ref):
        p = float(cand[np.argmin(np.abs(cand - ref))])
    elif cand.size:
        p = float(cand[0])
    else:
        p = closest_hr

    if np.abs(p - last5) > 10:
        p = last5

    picked.append(p)
    if np.isfinite(p):
        hist.append(p)

accel_df['peak'] = picked


In [8]:
MATCH_TOL_S = 15

acc_t = accel_df['accel_ts'].apply(_last_ts).to_numpy(dtype=float)
acc_peak = accel_df['peak'].to_numpy(dtype=float)
acc_raw = accel_df['peak_raw'].to_numpy(dtype=object)

valid = np.isfinite(acc_t) & np.isfinite(acc_peak)
order = np.argsort(acc_t[valid])
acc_t_s = acc_t[valid][order]
acc_peak_s = acc_peak[valid][order]
acc_raw_s = acc_raw[valid][order]

def _nearest(t):
    j = np.searchsorted(acc_t_s, t)
    cands = [k for k in (j - 1, j) if 0 <= k < acc_t_s.size]
    return min(cands, key=lambda k: abs(acc_t_s[k] - t)) if cands else None

def _cand_best_err(hr, m):
    c = np.asarray(acc_raw_s[m], dtype=float)
    c = c[np.isfinite(c)]
    return float(np.min(np.abs(c - hr))) if c.size else np.nan

err_df = hr_df.dropna(subset=['hr', 'hr_ts']).copy().sort_values('hr_ts').reset_index(drop=True)
matched = [_nearest(t) for t in err_df['hr_ts'].to_numpy(dtype=float)]
err_df['accel_peak']    = [acc_peak_s[m] if m is not None else np.nan for m in matched]
err_df['dt_s']          = [abs(acc_t_s[m] - t) / 1000 if m is not None else np.nan
                           for m, t in zip(matched, err_df['hr_ts'].to_numpy(dtype=float))]
err_df['abs_err']       = (err_df['accel_peak'] - err_df['hr']).abs()
err_df['cand_best_err'] = [_cand_best_err(h, m) if m is not None else np.nan
                           for h, m in zip(err_df['hr'], matched)]

# Keep only readings with an accel window within MATCH_TOL_MS
n_total = len(err_df)
err_df = err_df[err_df['dt_s'] <= MATCH_TOL_S].reset_index(drop=True)
print(f"kept {len(err_df)}/{n_total} HR readings within {MATCH_TOL_S:.0f}s of an accel window")

e = err_df['abs_err'].dropna()
cb = err_df['cand_best_err'].dropna()
summary = pd.DataFrame({
    'metric': ['n', 'mean', 'std', 'p50', 'p80', 'p90', 'p95', '%<=2bpm', '%<=5bpm'],
    'picked_peak': [len(e), e.mean(), e.std(),
                    e.quantile(0.50), e.quantile(0.80), e.quantile(0.90), e.quantile(0.95),
                    (e <= 2).mean() * 100, (e <= 5).mean() * 100],
    'best_candidate': [len(cb), cb.mean(), cb.std(),
                       cb.quantile(0.50), cb.quantile(0.80), cb.quantile(0.90), cb.quantile(0.95),
                       (cb <= 2).mean() * 100, (cb <= 5).mean() * 100],
}).round(2)

print(summary.to_string(index=False))

kept 1124/2540 HR readings within 15s of an accel window
 metric  picked_peak  best_candidate
      n      1124.00         1124.00
   mean         2.61            2.84
    std         2.35            2.87
    p50         1.83            2.00
    p80         4.00            4.83
    p90         6.00            6.83
    p95         8.00            8.22
%<=2bpm        59.07           58.19
%<=5bpm        85.85           83.36


In [ ]:
# Peak HR over time: PPG & accel smoothed (1-min median), firmware HR as-is.
# Timestamps are epoch ms (UTC); display in IST (Asia/Kolkata).
TZ = 'Asia/Kolkata'

def _last_ts(seq):
    return seq[-1] if len(seq) else np.nan

def _to_ist(ms):
    return pd.to_datetime(np.asarray(ms, dtype=float), unit='ms', utc=True).tz_convert(TZ)

# PPG peaks, aligned to the last timestamp of each green window
g_t = _to_ist(green_df['green_ts'].apply(_last_ts))
ppg_s = pd.Series(green_df['peak'].to_numpy(dtype=float), index=g_t).dropna().sort_index()
ppg_smooth = ppg_s.rolling('60s', center=True, min_periods=1).mean()

# Accel peaks, aligned to the last timestamp of each accel window
a_t = _to_ist(accel_df['accel_ts'].apply(_last_ts))
accel_s = pd.Series(accel_df['peak'].to_numpy(dtype=float), index=a_t).dropna().sort_index()
accel_smooth = accel_s.rolling('120s', center=True, min_periods=1).mean()

# Firmware HR (per block, carried forward) — plotted as-is
hr_s = pd.Series(hr_df['hr'].to_numpy(dtype=float),
                 index=_to_ist(hr_df['hr_ts'])).dropna().sort_index()
hr_smooth = hr_s.rolling('30s', center=True, min_periods=1).mean()

fig = go.Figure()
fig.add_trace(go.Scatter(x=hr_smooth.index, y=hr_smooth.to_numpy(), mode='lines',
                         name='Firmware HR', line=dict(color='black', width=2)))
fig.add_trace(go.Scatter(x=ppg_smooth.index, y=ppg_smooth.to_numpy(), mode='lines',
                         name='PPG HR', line=dict(color='crimson', width=2)))
fig.add_trace(go.Scatter(x=accel_smooth.index, y=accel_smooth.to_numpy(), mode='lines',
                         name='Accel HR', line=dict(color='royalblue', width=2)))
fig.update_layout(title='Predicted HR (smoothed) vs firmware HR',
                  xaxis_title='time (IST)', yaxis_title='HR (bpm)',
                  template='plotly_white', height=500)
fig.show()

In [ ]:
# Firmware HR vs PPG HR (green peak), keeping only green windows within
# GREEN_TOL_S of a reported firmware HR reading.
GREEN_TOL_S = 10.0

# reported firmware HR, sorted by time
hr_rep = hr_df.dropna(subset=['hr', 'hr_ts']).sort_values('hr_ts')
hr_ts_s = hr_rep['hr_ts'].to_numpy(dtype=float)
hr_val_s = hr_rep['hr'].to_numpy(dtype=float)

def _nearest_dt(t):
    """Gap (ms) from t to the nearest reported HR timestamp."""
    j = np.searchsorted(hr_ts_s, t)
    cands = [k for k in (j - 1, j) if 0 <= k < hr_ts_s.size]
    return min(abs(hr_ts_s[k] - t) for k in cands) if cands else np.inf

# green windows: value (PPG HR) + last timestamp
gp = green_df['peak'].to_numpy(dtype=float)
gt = green_df['green_ts'].apply(_last_ts).to_numpy(dtype=float)
ok = np.isfinite(gp) & np.isfinite(gt)
gp, gt = gp[ok], gt[ok]

near = np.array([_nearest_dt(t) for t in gt]) <= GREEN_TOL_S * 1000
print(f"kept {near.sum()}/{gt.size} green windows within {GREEN_TOL_S:.0f}s of an HR reading")

ppg_valid = pd.Series(gp[near], index=_to_ist(gt[near])).sort_index()
hr_line = pd.Series(hr_val_s, index=_to_ist(hr_ts_s)).sort_index()

fig = go.Figure()
fig.add_trace(go.Scatter(x=hr_line.index, y=hr_line.to_numpy(), mode='lines',
                         name='Firmware HR', line=dict(color='black', width=2)))
fig.add_trace(go.Scatter(x=ppg_valid.index, y=ppg_valid.to_numpy(), mode='markers',
                         name='PPG HR (valid windows)', marker=dict(color='crimson', size=5)))
fig.update_layout(title='Firmware HR vs PPG HR (windows within 10s of an HR reading)',
                  xaxis_title='time (IST)', yaxis_title='HR (bpm)',
                  template='plotly_white', height=500)
fig.show()
